In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- bisel8 on PepMSND, 3 seeds x 10 folds.
#
# PepMSND is a third pipeline, unrelated to the LoRA classification script:
# FULL-BACKBONE finetuning with a freeze schedule, a KAN fusion head, and ~140
# RDKit descriptors fed alongside the encoder. 10 pre-made folds, 640 molecules,
# 64 per test fold, pooled across folds for one MCC.
#
# BUDGET AND WHY THE SEEDS ARE SPLIT ACROSS ACCOUNTS. Their protocol is
# max_epochs 120 with patience 20 on val_mcc_best -- but val_mcc_best is MCC on 64
# molecules, where one flipped molecule moves the metric by ~0.03. A metric that
# noisy keeps hitting new maxima by chance, so a clean 20-epoch plateau is
# unlikely and most folds will run the full 120 epochs.
#
# At 120 epochs, on two T4s:
#     3 seeds (30 folds)  14.9 h   -> over the 12 h session limit
#     2 seeds (20 folds)   9.9 h   -> fits
#     1 seed  (10 folds)   5.0 h   -> comfortable
#
# Splitting keeps their epoch budget intact instead of cutting it to fit. Set
# SEEDS in Cell 5 per notebook. Results sync to Drive per fold under seed_<n>/, so
# the two accounts never collide and either notebook's Cell 6 pools whatever has
# landed so far.
#
# WHAT WE COMPARE AGAINST -- their shipped PepMSND predictions, 3 runs pooled over
# 10 folds:
#     PeptideMTR_lg        0.6613 +- 0.016
#     PeptideMLM-MTR_lg    0.6424 +- 0.022
#     PeptideMLM_lg        0.6180 +- 0.003   <- our teacher, the number that matters
#
# THE HYPERPARAMETER TRAP. Their script picks learning rates from a SUFFIX MATCH
# on the model name:
#     endswith("_lg")  -> backbone 7e-6, head 7e-4, freeze 2
#     fallback         -> backbone 1e-5, head 1e-3, freeze 3
# Their shipped runs used "aaronfeller_PeptideMLM_lg", which matches "_lg". Our
# folder is "peptideclm-2-mlm-bisel8", which matches NOTHING and would silently
# take the fallback -- different optimiser settings from the 0.6180 we are
# comparing to. So the three values are passed explicitly below.
subprocess.run('pip install -q -U "transformers>=5.0" lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time, shutil
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
NGPU = max(1, torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))


In [ ]:

# -- Cell 3 -- their code and data, our code, the teacher.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

SCRIPTS = REPO + "/training/03_PepMSND_training_code/scripts"
TRAIN_PY = SCRIPTS + "/train_pepmsnd_kan_paperstyle.py"
DATA_DIR = REPO + "/data/PepMSND_data"

if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

# kan.py sits next to the training script and is imported as "from kan import
# KANLinear". Running the script by path puts its own directory on sys.path, so
# the local kan.py wins over any pip package of the same name -- but only if we
# do not shadow it, hence cwd is set to SCRIPTS for every job below.
for p in (TRAIN_PY, SCRIPTS + "/kan.py", DATA_DIR + "/X_train1.csv",
          DATA_DIR + "/X_test10.csv", CODE + "/export_truncated.py"):
    assert os.path.exists(p), "missing: " + p
print("ok -- PepMSND pipeline present")


In [ ]:

# -- Cell 4 -- export bisel8.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)
KEEP = "0,1,2,3,5,6,10,16"
ARM = EXPORT + "/peptideclm-2-mlm-bisel8"

# Prefer the copy on Drive. The export is deterministic so deriving it here would
# give the identical model, but pulling a pinned artifact means both accounts run
# byte-identical weights and neither depends on export_truncated.py being current
# on Drive -- a stale copy there has already cost one run.
if not os.path.exists(ARM + "/model.safetensors"):
    os.makedirs(ARM, exist_ok=True)
    subprocess.run("rclone copy %s/models/peptideclm-2-mlm-bisel8 %s -P" % (REMOTE, ARM),
                   shell=True, check=False)
if os.path.exists(ARM + "/model.safetensors"):
    print("bisel8 <- Drive")
else:
    print("not on Drive, deriving from the teacher")
    r = subprocess.run(["python", "export_truncated.py", "--out", ARM, "--keep", KEEP],
                       cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-600:])
    if r.returncode != 0:
        print(r.stderr[-1200:])
    assert r.returncode == 0, "export failed"

KEEP_LIST = [0, 1, 2, 3, 5, 6, 10, 16]
kept = json.load(open(ARM + "/config.json")).get("pruned_from", {}).get("kept")
assert kept == KEEP_LIST, "wrong blocks in config: %s" % kept
# VERIFY THE WEIGHTS, NOT JUST THE CONFIG. A correct config.json sitting next to
# the wrong model.safetensors already cost this project four mislabelled
# benchmark runs. verify_weights compares exported block j against TEACHER block
# keep[j], tensor by tensor, so a swapped checkpoint cannot survive it.
_v = subprocess.run(["python", "-c",
                     "import sys; sys.path.insert(0, '.');"
                     "from export_truncated import verify_weights;"
                     "verify_weights(%r, %r, %r)" % (TEACH, ARM, KEEP_LIST)],
                    cwd=CODE, capture_output=True, text=True)
print(_v.stdout.strip() or _v.stderr[-800:])
assert _v.returncode == 0, (
    "THESE ARE NOT THE bisel8 WEIGHTS -- re-upload "
    "models/peptideclm-2-mlm-bisel8 to Drive.")

print("bisel8: %d blocks, %.1f MB"
      % (json.load(open(ARM + "/config.json"))["num_blocks"],
         os.path.getsize(ARM + "/model.safetensors") / 1e6))


In [ ]:

# -- Cell 5 -- 3 seeds x 10 folds. Resumable; checkpoints deleted as we go.
#
# Each job writes preds_fold{N}.csv into its own directory, plus a Lightning
# checkpoint. Thirty checkpoints at 339 MB would be 10.2 GB against a 20 GB quota,
# so each is deleted once its predictions exist -- the predictions are the only
# output anything downstream reads.
# ============================================================================
# SET THIS PER NOTEBOOK. Account A: [101, 202]   Account B: [303]
# Both write to the same Drive folder under seed_<n>/, so they cannot collide.
SEEDS = [101, 202]
# ============================================================================
FOLDS = list(range(1, 11))
OUT = WORK + "/results/pepmsnd_bisel8"
os.makedirs(OUT, exist_ok=True)

# Their published runs used these, via the "_lg" suffix branch. Passed explicitly
# because our folder name would otherwise hit the fallback (1e-5 / 1e-3 / 3).
BB_LR, HEAD_LR, FREEZE = "7e-6", "7e-4", "2"

subprocess.run("rclone copy %s/results/pepmsnd_bisel8 %s --transfers 8 -P"
               % (REMOTE, OUT), shell=True, check=False)
done0 = len(glob.glob(OUT + "/seed_*/fold_*/preds_fold*.csv"))
print("recovered %d completed folds from Drive" % done0)

jobs = [(s, f) for s in SEEDS for f in FOLDS]
todo = [(s, f) for s, f in jobs
        if not os.path.exists("%s/seed_%d/fold_%d/preds_fold%d.csv" % (OUT, s, f, f))]
print("%d jobs, %d to run" % (len(jobs), len(todo)))

running, free, t0, last = [], list(range(NGPU)), time.time(), time.time()
while todo or running:
    while todo and free:
        s, f = todo.pop(0); gpu = free.pop(0)
        d = "%s/seed_%d/fold_%d" % (OUT, s, f)
        os.makedirs(d, exist_ok=True)
        cmd = ["python", TRAIN_PY, "--model_name", ARM, "--fold", str(f),
               "--data_dir", DATA_DIR, "--save_path", d, "--seed", str(s),
               "--batch_size", "32",
               "--backbone_learning_rate", BB_LR,
               "--head_learning_rate", HEAD_LR,
               "--freeze_backbone_epochs", FREEZE]
        p = subprocess.Popen(cmd, cwd=SCRIPTS, stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((s, f, gpu, p, d))
        print("[%5.1f min] launch seed %d fold %2d  gpu%d" % ((time.time()-t0)/60, s, f, gpu))
    time.sleep(30)
    # Heartbeat: these jobs take ~20-30 min each, and silence for that long makes a
    # working run indistinguishable from a hung one.
    if time.time() - last > 600:
        last = time.time()
        print("   [%5.1f min] %d running, %d queued, %d done"
              % ((time.time()-t0)/60, len(running), len(todo),
                 len(glob.glob(OUT + "/seed_*/fold_*/preds_fold*.csv"))))
    for job in list(running):
        s, f, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and os.path.exists("%s/preds_fold%d.csv" % (d, f))
        print("[%5.1f min] seed %d fold %2d -> %s"
              % ((time.time()-t0)/60, s, f, "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            shutil.rmtree(os.path.join(d, "checkpoints"), ignore_errors=True)
            subprocess.run("rclone copy %s %s/results/pepmsnd_bisel8/seed_%d/fold_%d "
                           "--exclude 'checkpoints/**' --drive-chunk-size 64M"
                           % (d, REMOTE, s, f), shell=True, check=False)
        else:
            print("".join(open(d + "/train.log").readlines()[-15:]))
print("")
print("done in %.2f h" % ((time.time() - t0) / 3600))


In [ ]:

# -- Cell 6 -- pool the folds and compare.
#
# Their evaluation pools all 10 test folds into one prediction set (640 molecules)
# and computes a single MCC, which is how the 0.6180 in their shipped file is
# derived. Same here, per seed.
#
# NOTE the threshold: this pipeline emits PROBABILITIES and thresholds at 0.5,
# unlike the LoRA classification script which emits logits thresholded at 0.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

# Pull everything, so whichever notebook finishes last reports the full picture
# rather than only its own seeds.
subprocess.run("rclone copy %s/results/pepmsnd_bisel8 %s --transfers 8 -P"
               % (REMOTE, OUT), shell=True, check=False)
found = sorted({int(os.path.basename(d).split("_")[1])
                for d in glob.glob(OUT + "/seed_*")})
print("seeds present: %s   (this notebook ran %s)" % (found, SEEDS))

rows = []
for s in found:
    fs = sorted(glob.glob("%s/seed_%d/fold_*/preds_fold*.csv" % (OUT, s)))
    if not fs:
        continue
    d = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    y = d.true_label.values.astype(int)
    p = d.predicted_prob.values
    rows.append(dict(seed=s, folds=len(fs), n=len(d),
                     mcc=round(matthews_corrcoef(y, (p >= 0.5).astype(int)), 4),
                     auc=round(roc_auc_score(y, p), 4),
                     acc=round(accuracy_score(y, (p >= 0.5).astype(int)), 4)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
if len(res):
    print("\nbisel8 (84.8M, 8 blocks):  MCC %.4f +- %.4f over %d seeds"
          % (res.mcc.mean(), res.mcc.std(), len(res)))

print("\ntheir published PepMSND, 3 runs pooled over 10 folds:")
for k, v, e in [("PeptideMTR_lg  337M", 0.6613, 0.016),
                ("PeptideMLM-MTR_lg 337M", 0.6424, 0.022),
                ("PeptideMLM_lg  337M  <- our teacher", 0.6180, 0.003)]:
    print("   %-38s %.4f +- %.3f" % (k, v, e))
print("\ncontrols measured on this benchmark:")
print("   bag-of-tokens (405 counts)              0.4221")
print("   RDKit descriptors alone (~140 dims)     0.4871")
print("\nCAVEAT: PepMSND feeds the KAN head ~140 descriptors ALONGSIDE the encoder,")
print("and descriptors alone reach 0.4871. So the encoder carries less of this")
print("benchmark than of AmpHGT, and differences between backbones are compressed.")
print("Test folds are 64 molecules each; pooled n=640.")

res.to_csv(OUT + "/pepmsnd_bisel8_metrics.csv", index=False)
subprocess.run("rclone copy %s %s/results/pepmsnd_bisel8 --exclude 'checkpoints/**' "
               "--drive-chunk-size 64M -P" % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/pepmsnd_bisel8" % REMOTE)
